# 🚀 生成式AI應用開發｜第03週
## Gemini API 入門：從 API Key 到第一個 Python AI 函式

> **執行環境：Google Colab** ｜ **本版本使用 Gemini API**

本教材內容對應 OpenAI API 入門版，但改用 Google Gemini API。重點是讓你比較不同 LLM 平台在 Python SDK、API key、輸入格式、回應物件與 <font color="#1a73e8"><b>usage</b></font> 欄位上的差異。

官方文件入口：
- Gemini API Get started: https://ai.google.dev/gemini-api/docs/get-started
- Gemini text generation: https://ai.google.dev/gemini-api/docs/text-generation

<div style="border-left:4px solid #1a73e8; padding:10px 12px; background:#eef4ff; margin:10px 0;">
  <font color="#1a73e8"><b>本週任務</b></font><br>
  完成第一個 API 呼叫，並把呼叫流程封裝成可重複使用的 Python 函式。
</div>


> **學生版**：本檔保留課堂練習的 TODO 骨架。執行 Gemini API cells 可能產生成本或用量。

## 🎯 本週學習目標

| # | 能力 | Gemini 對應概念 |
|---|------|----------------|
| 1 | 安裝與匯入 Gemini SDK | `google-genai` |
| 2 | 使用 Colab Secrets 讀取 API key | `GEMINI_API_KEY` |
| 3 | 建立 Gemini client | `genai.Client()` |
| 4 | 呼叫 Interactions API | `client.interactions.create()` |
| 5 | 使用 `system_instruction` 與 `input` | 角色設定與使用者輸入 |
| 6 | 解析 `interaction.output_text` 與 <font color="#1a73e8"><b>usage</b></font> | 顯示回答、除錯、估算成本 |
| 7 | 封裝函式與錯誤處理 | 後續 App 開發 |


---
## 🧰 第一節：安裝套件

In [ ]:
!pip install -U google-genai --quiet
print("Google GenAI SDK 安裝完成")

In [ ]:
import os
import json
from google import genai

print("模組載入完成")

---
## 🔐 第二節：設定 API Key

請到 Google AI Studio 建立 API key，並在 Colab Secrets 新增：

- Name: `GEMINI_API_KEY`
- Value: 你的 Gemini API key
- 開啟 Notebook access

<font color="#b00020"><b>不要把真正的 API key</b></font> 寫在 notebook 裡。

<div style="border-left:4px solid #f9ab00; padding:10px 12px; background:#fff8e1; margin:10px 0;">
  <font color="#b06000"><b>安全提醒</b></font><br>
  <font color="#b00020"><b>API key 是機密資訊</b></font>。請使用 Colab Secrets 或環境變數，不要直接寫在 notebook、作業或截圖中。
</div>


In [ ]:
try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
    if api_key:
        os.environ["GEMINI_API_KEY"] = api_key
        print(f"已讀取 Gemini API key（前 8 碼）：{api_key[:8]}...")
    else:
        print("Secrets 中找不到 GEMINI_API_KEY，請先完成設定")
except Exception as e:
    print(f"目前可能不是在 Colab 執行：{e}")
    print("若在本機執行，請先設定環境變數 GEMINI_API_KEY")

print("目前讀取狀態：", "已設定" if os.getenv("GEMINI_API_KEY") else "未設定")

### 💻 本機環境變數設定方式

```powershell
$env:GEMINI_API_KEY="你的 Gemini API key"
setx GEMINI_API_KEY "你的 Gemini API key"
```

```bash
export GEMINI_API_KEY="你的 Gemini API key"
```

---
## 🔌 第三節：建立 Gemini Client

`genai.Client()` 會從環境變數讀取 `GEMINI_API_KEY`。

In [ ]:
client = genai.Client()

# 課堂預設使用 Gemini Flash。若老師課前確認其他模型更適合，可用 GEMINI_MODEL 覆蓋。
MODEL = os.getenv("GEMINI_MODEL", "gemini-3.5-flash")

print(f"Client 建立完成，使用模型：{MODEL}")

---
## ✨ 第四節：第一個 Gemini API 呼叫

Gemini Interactions API 使用 `client.interactions.create()`，並以 `interaction.output_text` 取出文字輸出。

In [ ]:
interaction = client.interactions.create(
    model=MODEL,
    input="請用繁體中文，用三句話解釋什麼是生成式 AI。"
)

print(interaction.output_text)

### 🔍 4-1 檢查 interaction 物件

實務開發時，不只要看回答，也要觀察 id、model、<font color="#1a73e8"><b>usage</b></font> 等欄位。

In [ ]:
print("interaction id:", interaction.id)
print("model:", interaction.model)
print("output_text:")
print(interaction.output_text)

print()
print("usage:")
print(interaction.usage)

---
## 🧭 第五節：加入 system_instruction

Gemini 使用 `system_instruction` 設定模型行為，概念上類似 OpenAI 版本的 `instructions`。它適合放角色、語氣、格式限制與穩定規則。

In [ ]:
interaction = client.interactions.create(
    model=MODEL,
    system_instruction="你是一位資管系 Python 助教。回答要精簡、具體，並使用繁體中文。",
    input="請解釋 API key 為什麼不能直接寫在程式碼裡。"
)

print(interaction.output_text)

---
## 💬 第六節：多輪對話與 previous_interaction_id

Gemini Interactions API 的多輪對話主線是使用 `previous_interaction_id`，讓伺服器延續前一次 interaction 的上下文。

這和 OpenAI / Claude 常見的 messages list 不同：

| 平台 | 多輪對話常見做法 |
|---|---|
| OpenAI Responses API | 可用 role/content list 或 previous response |
| Gemini Interactions API | 使用 `previous_interaction_id` 延續對話 |
| Claude Messages API | 每次送出完整 messages history |


In [ ]:
interaction1 = client.interactions.create(
    model=MODEL,
    input="我正在學習 Python API 開發。"
)
print("第一輪：")
print(interaction1.output_text)

interaction2 = client.interactions.create(
    model=MODEL,
    input="根據上一句，請建議我下一步該練什麼。",
    previous_interaction_id=interaction1.id
)
print()
print("第二輪：")
print(interaction2.output_text)

### 🧠 6-1 什麼時候使用 system_instruction？

1. 規則固定、每次都要遵守時，放在 `system_instruction`。
2. 使用者這次提出的問題，放在 `input`。
3. 要延續上一輪對話時，使用 `previous_interaction_id`。
4. 做聊天 App 時，需記錄上一輪 interaction id，或自行管理歷史。

---
## 🧩 第七節：封裝成函式

In [ ]:
def ask_ai(user_input, role="你是一位有幫助的助理", model=MODEL):
    # TODO 1：呼叫 client.interactions.create()
    # TODO 2：傳入 model、system_instruction、input
    # TODO 3：回傳 interaction.output_text
    return "尚未完成 ask_ai()" 

---
## 🛡️ 第八節：基本錯誤處理

In [ ]:
def ask_ai_safe(user_input, role="你是一位有幫助的助理", model=MODEL):
    try:
        # TODO 1：呼叫 Gemini API
        # TODO 2：成功時回傳 (True, interaction.output_text)
        return False, "尚未完成 ask_ai_safe()"
    except Exception as e:
        return False, str(e)


success, result = ask_ai_safe("請用一句話說明 JSON 是什麼。")
print(result)

---
## 📊 第九節：讀取 <font color="#1a73e8"><b>usage</b></font> 並建立除錯資訊

In [ ]:
def ask_ai_with_metadata(user_input, role="你是一位有幫助的助理", model=MODEL):
    # TODO 1：呼叫 API
    # TODO 2：回傳 dict，包含 id、model、answer、usage
    return {"id": None, "model": model, "answer": "尚未完成", "usage": None}


result = ask_ai_with_metadata("請列出學習 Gemini API 前需要會的 3 個 Python 技能。")
print(json.dumps(result, ensure_ascii=False, indent=2, default=str))

---
## 🧪 第十節：課堂練習

### 📝 練習 A：Python 作業批改助教

請讓 Gemini 扮演 Python 作業批改助教，檢查一段縮排錯誤的程式，並輸出「錯誤原因、修正方式、修正版程式碼」。

<div style="border-left:4px solid #188038; padding:10px 12px; background:#e6f4ea; margin:10px 0;">
  <font color="#188038"><b>課堂練習</b></font><br>
  先完成練習 A，再依時間進行練習 B 與選做練習 C。
</div>


In [ ]:
student_code = '''
def add_numbers(a, b):
return a + b

print(add_numbers(3, 5))
'''

role = ""
prompt = ""
success, result = False, "尚未完成課堂練習"
print(result)

---
### 💰 練習 B：<font color="#1a73e8"><b>usage</b></font> 與費用估算

請讀取 `<font color="#1a73e8"><b>usage</b></font>`，練習把 token 使用量整理成 dict。價格會變動，正式估價前請查官方 pricing。

<div style="border-left:4px solid #f9ab00; padding:10px 12px; background:#fff8e1; margin:10px 0;">
  <font color="#b06000"><b>成本提醒</b></font><br>
  本節重點是學會讀取 <font color="#1a73e8"><b>usage</b></font> 與建立估算流程；實際價格請以上課當天官方 pricing 為準。
</div>


In [ ]:
def summarize_usage(usage):
    # TODO：整理 total_input_tokens、total_output_tokens、total_tokens
    return {"status": "尚未完成"}


result = ask_ai_with_metadata("請用兩句話說明為什麼 API 呼叫需要注意成本。")
print(result["answer"])
print(summarize_usage(result["usage"]))

---
### 🎨 練習 C（選做）：自由設計一個角色

自訂至少兩個 `system_instruction`，觀察同一個問題在不同角色設定下的回答差異。

In [ ]:
question = "我想做一個生成式 AI 期末專題，可以從哪裡開始？"
roles = ["", ""]

for i, role in enumerate(roles, 1):
    print(f"===== 角色 {i} =====")
    print("尚未完成")
    print()

---
## ✅ 本週小結

| 技能 | Gemini 寫法 |
|------|-------------|
| API key | `GEMINI_API_KEY` |
| Client | `client = genai.Client()` |
| 文字生成 | `client.interactions.create()` |
| 角色設定 | `system_instruction` |
| 文字輸出 | `interaction.output_text` |
| 多輪對話 | `previous_interaction_id` |
